# Spotify Hit Predictor — SQL Exploration

**Objective:** query the cleaned data set with SQL to find patterns in popularity, before modeling. There is where H1 ("a dectectable pattern exists") and H2("genre is secondary") 
get their first test.

**Input:** `tracks_clean.csv` — 112,782 tracks, 114 genres (output of notebook
01, where cleaning and validation are documented).

**This notebook produces:** genre popularity rankings, a hit-rate analysis by
genre (popularity ≥ 70 threshold), and a cross-genre duplicate count flagged
as a data-leakage risk for the modeling phase.

In [2]:
import pandas as pd 
import sqlite3

df = pd.read_csv('../data/tracks_clean.csv')
print("Numbers of songs loaded:", df.shape[0])


Numbers of songs loaded: 112782


## Loading the data into SQLite

The cleaned dataset (`tracks_clean.csv`) is loaded into a SQLite database so
the analysis can be done in SQL

In [3]:
# Connect to the database 
conn = sqlite3.connect('spotify.db')

# Cleanned DataFrame put into a 'tracks' table
df.to_sql('tracks', conn, if_exists='replace', index='False')
print('tracks table created in spotify.db' )

tracks table created in spotify.db


In [4]:
query = 'SELECT COUNT(*) AS total_songs FROM tracks'
pd.read_sql(query, conn)

,total_songs
0,112782


## Q8 — Which genres have the highest average popularity?

Grouping by genre and averaging popularity. `MAX(popularity)` is included to
catch the average-vs-peak paradox: a genre can rank low on average yet contain
the single most popular track.

In [5]:
query = """ 
SELECT track_genre, 
     COUNT(*) AS songs,
     ROUND(AVG(popularity), 1) AS popularity_avg,
     MAX(popularity) AS popularity_max
FROM tracks 
GROUP BY track_genre
ORDER BY popularity_avg DESC
LIMIT 15
"""

pd.read_sql(query, conn)


,track_genre,songs,popularity_avg,popularity_max
0,pop-film,998,59.3,80
1,k-pop,993,57.0,88
2,chill,999,53.7,93
3,sad,1000,52.4,83
4,grunge,999,49.6,85
5,indian,993,49.5,88
6,anime,999,48.8,83
7,emo,999,48.1,87
8,sertanejo,1000,47.9,63
9,pop,993,47.9,100


**Result:** pop-film (59.3), k-pop (57.0), chill (53.7) lead. The spread across
the top genres is narrow — about 15 points.

**Finding (H2):** if genre were a strong predictor of popularity, we'd expect a
wide gap between "popular" and "unpopular" genres. Instead the averages cluster
tightly. Early evidence that the genre label carries limited predictive signal.

### Inspecting the 'piano' genre

Checking what actually sits inside a non-standard genre label. `piano` is an
instrument, not a genre, this confirms theinconsistency documented
in notebook 01 (Q7).

In [6]:
query = """
SELECT track_name, artists, popularity, duration_ms/60000.0 AS min
FROM tracks
WHERE track_genre = 'piano'
ORDER BY popularity DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,track_name,artists,popularity,min
0,I Ain't Worried,OneRepublic,96,2.474750
1,Running Up That Hill (A Deal With God),Kate Bush,90,4.982217
2,Hold Me Closer,Elton John;Britney Spears,89,3.370750
3,Somewhere Only We Know,Keane,85,3.952433
4,Running Up That Hill (A Deal With God) - 2018 ...,Kate Bush,85,5.014000
5,I'm Still Standing,Elton John,84,3.057333
6,Counting Stars,OneRepublic,83,4.287767
7,Sunshine,OneRepublic,83,2.730900
8,"Rocket Man (I Think It's Going To Be A Long, L...",Elton John,81,4.693550
9,How to Save a Life,The Fray,80,4.375550


**Result:** the top 'piano' tracks are pop and rock hits.
OneRepublic's "I Ain't Worried", Kate Bush's "Running Up That Hill", Billy Joel's "Uptown Girl", multiple Elton John songs. Not one is an instrumental piano piece.
These are vocal mainstream songs that happen to feature piano in the arrangement.

**Finding (H2):** 'piano' is an instrument label, not a genre. A pop song and a rock song sit side by side under it because they share an instrument, not a
sound category. This is direct evidence that track_genre mixes taxonomies — the same inconsistency documented in notebook 01 (Q7), now shown concretely.
If the label can't even separate pop from rock, it's a weak predictive variable by construction.

## Q10 — Unique songs vs total rows (cross-genre duplicates)

Notebook 01 flagged that duplicates were likely the same track appearing under multiple genres. This query confirms it directly: GROUP_CONCAT lists every
genre each track_id is tagged with.

In [7]:
query = """
SELECT track_name, artists, COUNT(*) AS count, 
       GROUP_CONCAT(track_genre, ', ') AS genres
FROM tracks
GROUP BY track_id
HAVING count > 1
ORDER BY count DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,track_name,artists,count,genres
0,Baby Blue - Remastered 2010,Badfinger,9,"blues, country, folk, j-pop, j-rock, power-pop..."
1,Layla,Derek & The Dominos,8,"blues, british, country, folk, hard-rock, psyc..."
2,Layla,Derek & The Dominos,8,"blues, british, country, folk, hard-rock, psyc..."
3,Liggi,Ritviz,7,"edm, hip-hop, indian, indie-pop, indie, pop-fi..."
4,Mountain Song,Jane's Addiction,7,"alt-rock, blues, funk, grunge, hard-rock, meta..."
5,Trouble No More,Allman Brothers Band,7,"blues, country, folk, hard-rock, j-rock, singe..."
6,Let Me Hear,"Fear, and Loathing in Las Vegas",7,"hard-rock, hardcore, j-pop, j-rock, metal, met..."
7,Never Gonna Give You Up,The Black Keys,7,"alt-rock, alternative, blues, garage, punk-roc..."
8,Udd Gaye,Ritviz,7,"edm, hip-hop, indian, indie-pop, indie, pop-fi..."
9,Show Me The Way,Peter Frampton,7,"blues, british, country, folk, hard-rock, sing..."


**Result:** the most-tagged tracks appear under 7–9 genres each. A single recording is labeled across genres that share almost nothing sonically.

This is the mechanism behind the 21% duplicate rate (89,023 unique tracks in112,782 rows): one recording, many genre tags.

**Dual reading of this artifact:**
- **As business signal (Q24):** a track tagged across more genres has more
  playlist surface — more chances to be surfaced by Spotify's collaborative
  filtering. The versatility analysis will use these duplicate rows on purpose.
- **As technical risk (Week 4):** the same recording landing in both the train
  and test split would inflate the model's apparent accuracy. These duplicates
  will be removed before training (data leakage prevention).

Same artifact, opposite treatment — signal for the versatility question, noise for the model.

**Finding (H2):** if one recording carries 9 contradictory genre labels, the label cannot be describing the sound. The measurable audio is the more reliable
variable — which is exactly what H2 predicts.

Now the scale. The previous query showed *which* tracks repeat; this one counts *how many* across the whole table.

In [8]:
query = """ 
SELECT COUNT(*) AS total_rows ,
       COUNT(DISTINCT track_id) AS unique_songs,
       COUNT(*) - COUNT(DISTINCT track_id) AS duplicates 
FROM tracks 
"""

pd.read_sql(query, conn)

,total_rows,unique_songs,duplicates
0,112782,89023,23759


**Result:** 112,782 total rows, 89,023 unique songs → 23,759 duplicate rows (21% of the dataset).

## Q11, Q14 — Hit rate by genre, and defining the "hit" threshold

A "hit" needs a popularity cutoff, and that choice shouldn't be arbitrary. This counts hits per genre at threshold 70. The same query is then re-run at
60 (below) to test whether the ranking is stable. 

In [9]:
query = """
SELECT track_genre, 
      COUNT(*) AS songs, 
      SUM(CASE WHEN popularity >= 70 THEN 1 ELSE 0 END) AS hits,
      MAX(popularity) AS popularity_max
FROM tracks 
GROUP BY track_genre 
ORDER BY hits DESC
LIMIT 15
"""

pd.read_sql(query, conn)


,track_genre,songs,hits,popularity_max
0,pop,993,317,100
1,dance,965,244,100
2,electro,998,241,89
3,k-pop,993,225,88
4,house,999,221,90
5,metal,995,217,88
6,rock,1000,198,96
7,indie,997,185,92
8,edm,993,181,98
9,indie-pop,1000,177,88


In [10]:
query = """
SELECT track_genre, 
      COUNT(*) AS songs, 
      SUM(CASE WHEN popularity >= 60 THEN 1 ELSE 0 END) AS hits,
      MAX(popularity) AS popularity_max
FROM tracks 
GROUP BY track_genre 
ORDER BY hits DESC
LIMIT 15
"""

pd.read_sql(query, conn)


,track_genre,songs,hits,popularity_max
0,pop,993,644,100
1,pop-film,998,530,80
2,k-pop,993,502,88
3,metal,995,471,88
4,electro,998,450,89
5,house,999,411,90
6,hip-hop,990,408,99
7,edm,993,376,98
8,hard-rock,998,360,88
9,indie-pop,1000,352,88


**Result (threshold 70):** pop leads with 317 hits, followed by dance (244) and electro (241). The dataset is balanced by genre and with ~1,000 songs per genre, so hit counts are directly comparable
without normalizing.

## Q9 — Does 'pop' dominate?
It depends entirely on how we measure.

- **By average popularity (Q8):** pop ranks #9. Mediocre.
- **By number of hits (Q11):** pop ranks #1 — 317 hits, including a track 
  scoring a perfect 100 (though that same recording, "Unholy," is also 
  tagged under dance).

Both are true. Pop is a volume factory: it releases so many songs that most are forgettable (dragging the average down to #9) yet it still produces more
hits in absolute terms than any other genre. A niche genre like k-pop is more consistent (higher average) but generates fewer absolute hits.

**Finding (H2):** the genre label "pop" simultaneously means the single best track in the dataset and a mass of mediocre ones. It cannot predict whether
*a given song* will be a hit — it's too broad and internally contradictory.
This is precisely why the model should lean on measurable audio features
rather than the genre tag.

In [11]:
query = """
SELECT track_name, artists, track_genre, popularity
FROM tracks
WHERE popularity = 100
"""

pd.read_sql(query, conn)

,track_name,artists,track_genre,popularity
0,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,dance,100
1,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,pop,100


**Note:** "pop has 317 hits" means 317 hit-rows under the pop label not 317 unique songs. Given the 21% cross-genre duplication (Q10),
some of these hits are the same recording counted under multiple genres, as "Unholy" is. The ranking holds, but the honest phrasing is "hit-rows per
label," not "unique hits per genre."

## Q12 - Are short songs more popular than long ones?

In [24]:
query = """
SELECT
   CASE 
     WHEN duration_ms < 180000 THEN 'short (<3min)'
     WHEN duration_ms < 240000 THEN 'medium (3-4 min)'
     ELSE 'long (4min+)'
   END AS duration_group,
   COUNT(*) AS song, 
   ROUND (AVG(popularity), 1) AS popularity_avg
FROM tracks
GROUP BY 
   CASE 
        WHEN duration_ms < 180000 THEN 'short (<3min)'
        WHEN duration_ms < 240000 THEN 'medium (3-4 min)'
        ELSE 'long (4min+)'
    END 
ORDER BY popularity_avg DESC
"""

pd.read_sql(query, conn)

,duration_group,song,popularity_avg
0,medium (3-4 min),42261,34.8
1,long (4min+),38531,33.4
2,short (<3min),31990,31.5


**Result (naive, not deduplicated):** medium (34.8) leads, followed by 
long (33.4) and short (31.5) — not a clean "longer = more popular" 
trend. Medium-length songs (3-4 min) actually outperform both long and 
short ones in this raw pass. 



In [13]:
query = """
SELECT 
  track_name, 
  artists, 
  track_genre,
  duration_ms / 60000.0 AS duration_min,
  popularity
FROM tracks
WHERE duration_ms > 240000
ORDER BY popularity DESC
LIMIT 15 
"""

pd.read_sql(query, conn)

,track_name,artists,track_genre,duration_min,popularity
0,Tití Me Preguntó,Bad Bunny,latin,4.061933,97
1,Tití Me Preguntó,Bad Bunny,latino,4.061933,97
2,Tití Me Preguntó,Bad Bunny,reggae,4.061933,97
3,Tití Me Preguntó,Bad Bunny,reggaeton,4.061933,97
4,Ojitos Lindos,Bad Bunny;Bomba Estéreo,latin,4.304967,95
5,Ojitos Lindos,Bad Bunny;Bomba Estéreo,latino,4.304967,95
6,Moscow Mule,Bad Bunny,latin,4.098983,94
7,Moscow Mule,Bad Bunny,latino,4.098983,94
8,Ojitos Lindos,Bad Bunny;Bomba Estéreo,reggae,4.304967,94
9,Moscow Mule,Bad Bunny,reggae,4.098983,94


In [14]:
query = """
SELECT 
  track_name, 
  artists, 
  track_genre,
  duration_ms / 60000.0 AS duration_min,
  popularity
FROM tracks
WHERE duration_ms < 180000
ORDER BY popularity DESC
LIMIT 15 
"""

pd.read_sql(query, conn)

,track_name,artists,track_genre,duration_min,popularity
0,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,dance,2.615717,100
1,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,pop,2.615717,100
2,I'm Good (Blue),David Guetta;Bebe Rexha,dance,2.920633,98
3,I'm Good (Blue),David Guetta;Bebe Rexha,edm,2.920633,98
4,La Bachata,Manuel Turizo,latin,2.710617,98
5,La Bachata,Manuel Turizo,latino,2.710617,98
6,I'm Good (Blue),David Guetta;Bebe Rexha,pop,2.920633,98
7,La Bachata,Manuel Turizo,reggae,2.710617,98
8,La Bachata,Manuel Turizo,reggaeton,2.710617,98
9,Me Porto Bonito,Bad Bunny;Chencho Corleone,latin,2.976117,97


Inspected the top 15 tracks behind both the "long" and 
"short" groups. Both were dominated by a handful of songs repeated 3-4x 
each. The same track appearing once per genre tag (e.g., "Tití Me 
Preguntó" in latin/latino/reggae/reggaeton, "Unholy" in dance/pop). The 
naive averages above are inflated by this repetition, not by duration 
itself; a few multi-tagged super-hits are being counted multiple times 
within their duration group.

In [23]:
query = """
SELECT 
  CASE 
    WHEN duration_ms < 180000 THEN 'short (<3min)'
    WHEN duration_ms < 240000 THEN 'medium (3-4min)'
    ELSE 'long (4min+)'
  END AS duration_group,
  COUNT(*) AS unique_songs,
  ROUND(AVG(popularity), 1) AS popularity_avg
FROM (
  SELECT DISTINCT track_id, duration_ms, popularity
  FROM tracks
) AS unique_tracks
GROUP BY
  CASE 
    WHEN duration_ms < 180000 THEN 'short (<3min)'
    WHEN duration_ms < 240000 THEN 'medium (3-4min)'
    ELSE 'long (4min+)'
  END
ORDER BY popularity_avg DESC

"""


pd.read_sql(query, conn)

,duration_group,unique_songs,popularity_avg
0,medium (3-4min),32468,35.0
1,long (4min+),31227,32.9
2,short (<3min),26047,31.6


**First deduplication attempt:** grouped by duration using `SELECT 
DISTINCT track_id, duration_ms, popularity`. Result: medium (35.0) > long 
(32.9) > short (31.6) — song counts dropped substantially from the naive 
pass (e.g., short: 31,990 → 26,047), suggesting duplicates were being 
removed. The medium > long > short order held, but this looked like a 
clean fix — until spot-checking individual tracks (below) revealed it 
wasn't fully deduplicating.

In [16]:
query = """

SELECT DISTINCT track_id, track_name, artists, duration_ms / 60000.0 AS duration_min, popularity
FROM tracks
WHERE duration_ms > 240000
ORDER BY popularity DESC
LIMIT 15

"""

pd.read_sql(query, conn)

,track_id,track_name,artists,duration_min,popularity
0,1IHWl5LamUGEuP4ozKQSXZ,Tití Me Preguntó,Bad Bunny,4.061933,97
1,3k3NWokhRRkEPhCzPmV8TW,Ojitos Lindos,Bad Bunny;Bomba Estéreo,4.304967,95
2,6Xom58OOXk2SoU711L2IXO,Moscow Mule,Bad Bunny,4.098983,94
3,3k3NWokhRRkEPhCzPmV8TW,Ojitos Lindos,Bad Bunny;Bomba Estéreo,4.304967,94
4,2QjOHCTQ1Jl3zawyYOpxh6,Sweater Weather,The Neighbourhood,4.006667,93
5,3JvKfv6T31zO0ini8iNItO,Another Love,Tom Odell,4.072667,93
6,75FEaRjZTKLhTrFGsfMUXR,Running Up That Hill (A Deal With God),Kate Bush,4.982217,90
7,1cKHdTo9u0ZymJdPGSh6nq,I Was Never There,The Weeknd;Gesaffelstein,4.017767,90
8,1DIXPcTDzTj8ZMHt3PDt8p,Gangsta's Paradise,Coolio;L.V.,4.011550,89
9,5FVd6KXrgO9B3JPmC8OPst,Do I Wanna Know?,Arctic Monkeys,4.539900,88


In [17]:
query = """
SELECT *
FROM tracks
WHERE track_id = '3k3NWokhRRkEPhCzPmV8TW'
"""
pd.read_sql(query, conn)

,False,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,66739,3k3NWokhRRkEPhCzPmV8TW,Bad Bunny;Bomba Estéreo,Un Verano Sin Ti,Ojitos Lindos,95,258298,0,0.647,0.686,...,-5.745,0,0.0413,0.08,0.000001,0.528,0.268,79.928,4,latin
1,67581,3k3NWokhRRkEPhCzPmV8TW,Bad Bunny;Bomba Estéreo,Un Verano Sin Ti,Ojitos Lindos,95,258298,0,0.647,0.686,...,-5.745,0,0.0413,0.08,0.000001,0.528,0.268,79.928,4,latino
2,87485,3k3NWokhRRkEPhCzPmV8TW,Bad Bunny;Bomba Estéreo,Un Verano Sin Ti,Ojitos Lindos,94,258298,0,0.647,0.686,...,-5.745,0,0.0413,0.08,0.000001,0.528,0.268,79.928,4,reggae
3,88484,3k3NWokhRRkEPhCzPmV8TW,Bad Bunny;Bomba Estéreo,Un Verano Sin Ti,Ojitos Lindos,94,258298,0,0.647,0.686,...,-5.745,0,0.0413,0.08,0.000001,0.528,0.268,79.928,4,reggaeton


In [18]:
query = """
SELECT track_id, track_name, artists, 
       duration_ms / 60000.0 AS duration_min,
       MAX(popularity) AS popularity
FROM tracks
GROUP BY track_id 
ORDER BY popularity DESC
LIMIT 15

"""
pd.read_sql(query, conn)

,track_id,track_name,artists,duration_min,popularity
0,3nqQXoyQOWXiESFLlDF1hG,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,2.615717,100
1,2tTmW7RDtMQtBk7m2rYeSw,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,3.315617,99
2,5ww2BF9slyYgNOk37BlC4u,La Bachata,Manuel Turizo,2.710617,98
3,4uUG5RXrOk84mYEfFvj3cK,I'm Good (Blue),David Guetta;Bebe Rexha,2.920633,98
4,6Sq7ltF9Qa7SNFBsV5Cogx,Me Porto Bonito,Bad Bunny;Chencho Corleone,2.976117,97
5,1IHWl5LamUGEuP4ozKQSXZ,Tití Me Preguntó,Bad Bunny,4.061933,97
6,5IgjP7X4th6nMNDh4akUHb,Under The Influence,Chris Brown,3.076883,96
7,5Eax0qFko2dh7Rl2lYs3bx,Efecto,Bad Bunny,3.551017,96
8,4h9wh7iOZ0GGn8QVp4RAOB,I Ain't Worried,OneRepublic,2.474750,96
9,4LRPiXqCikLlN15c3yImP7,As It Was,Harry Styles,2.788383,95


**Result (deduplicated, full dataset):** the true top-15 tracks are 
dominated by short, beat-driven songs — Unholy (100), Quevedo Bzrp (99), 
La Bachata (98), I'm Good (98), Me Porto Bonito (97), Tití Me Preguntó 
(97) — reversing the naive "medium/long wins" finding entirely. 11 of 15 
top tracks run under 4 minutes.

In [19]:
query = """
SELECT 
  CASE 
    WHEN duration_ms < 180000 THEN 'short (<3min)'
    WHEN duration_ms < 240000 THEN 'medium (3-4min)'
    ELSE 'long (4min+)'
  END AS duration_group,
  COUNT(*) AS unique_songs,
  ROUND(AVG(popularity), 1) AS popularity_avg
FROM (
  SELECT track_id, duration_ms, MAX(popularity) AS popularity
  FROM tracks
  GROUP BY track_id
) AS unique_tracks
GROUP BY
  CASE 
    WHEN duration_ms < 180000 THEN 'short (<3min)'
    WHEN duration_ms < 240000 THEN 'medium (3-4min)'
    ELSE 'long (4min+)'
  END
ORDER BY popularity_avg DESC
"""

pd.read_sql(query, conn)

,duration_group,unique_songs,popularity_avg
0,medium (3-4min),32178,35.0
1,long (4min+),30994,32.9
2,short (<3min),25851,31.6


**Result (fully deduplicated, GROUP BY track_id + MAX):** medium (35.0) 
> long (32.9) > short (31.6). Song counts dropped slightly further than 
the DISTINCT attempt (e.g., short: 26,047 → 25,851) — confirming 
GROUP BY + MAX caught a small number of near-duplicates that DISTINCT 
missed (like "Ojitos Lindos"). 

**This is the final, trustworthy answer:** even after full deduplication, 
songs in the 3-4 minute range outperform both longer and shorter tracks 
on average — not a simple "longer is better" pattern.

In [20]:
query = """
SELECT track_id, track_name, artists, 
       duration_ms / 60000.0 AS duration_min,
       MAX(popularity) AS popularity
FROM tracks
WHERE duration_ms > 240000
GROUP BY track_id 
ORDER BY popularity DESC
LIMIT 15
"""

pd.read_sql(query, conn)

,track_id,track_name,artists,duration_min,popularity
0,1IHWl5LamUGEuP4ozKQSXZ,Tití Me Preguntó,Bad Bunny,4.061933,97
1,3k3NWokhRRkEPhCzPmV8TW,Ojitos Lindos,Bad Bunny;Bomba Estéreo,4.304967,95
2,6Xom58OOXk2SoU711L2IXO,Moscow Mule,Bad Bunny,4.098983,94
3,3JvKfv6T31zO0ini8iNItO,Another Love,Tom Odell,4.072667,93
4,2QjOHCTQ1Jl3zawyYOpxh6,Sweater Weather,The Neighbourhood,4.006667,93
5,75FEaRjZTKLhTrFGsfMUXR,Running Up That Hill (A Deal With God),Kate Bush,4.982217,90
6,1cKHdTo9u0ZymJdPGSh6nq,I Was Never There,The Weeknd;Gesaffelstein,4.017767,90
7,1DIXPcTDzTj8ZMHt3PDt8p,Gangsta's Paradise,Coolio;L.V.,4.011550,89
8,7lQ8MOhq6IN2w8EYcFNSUk,Without Me,Eminem,4.838667,88
9,7fBv7CLKzipRk6EC6TWHOB,The Hills,The Weeknd,4.037550,88


## Q12 — Final Answer
Duration does relate to popularity, but not in a simple linear way — the 
relationship isn't "longer is better" or "shorter is better." Songs in 
the 3-4 minute range consistently outperform both longer and shorter 
tracks (35.0 vs 32.9 vs 31.6), even after full deduplication. This held 
up across all three query versions (naive, DISTINCT, GROUP BY), so it's 
not a duplication artifact — it's a real, if modest, pattern.

What's more interesting than the averages: duration also correlates with 
*how* a song achieves hit status. Short hits lean beat-driven and 
sonically uniform (dance, reggaeton). Long hits are genre-diverse, often 
winning through narrative or emotional build rather than rhythm. Carried 
into Week 3: test whether duration correlates with valence or 
acousticness.

## Q13 — Do certain artists consistently dominate hits?

Deduplicated by track_id first (same subquery pattern as Q12) to avoid 
genre-repetition inflating any single artist's song or hit count.

In [21]:
query = """
SELECT artists, 
       COUNT(*) AS songs, 
       SUM(CASE WHEN popularity >= 70 THEN 1 ELSE 0 END) AS hits, 
       ROUND(AVG(popularity), 1) AS popularity_avg
FROM (
  SELECT track_id, artists, MAX(popularity) AS popularity
  FROM tracks
  GROUP BY track_id   
) AS unique_tracks
GROUP BY artists
ORDER BY hits DESC 
LIMIT 15
"""

pd.read_sql(query, conn)

,artists,songs,hits,popularity_avg
0,BTS,143,52,67.9
1,Billie Eilish,49,27,44.8
2,Bad Bunny,22,22,85.4
3,XXXTENTACION,67,20,62.4
4,BLACKPINK,40,20,69.8
5,Arctic Monkeys,110,20,60.1
6,Stray Kids,22,19,73.1
7,Adele,49,19,65.1
8,The Neighbourhood,25,18,73.9
9,OneRepublic,124,16,35.9




**Finding (two distinct strategies among top artists):**
- **Volume strategy:** BTS (143 songs → 52 hits, 36% rate), The Beatles 
  (149 → 14, 9%), OneRepublic (124 → 16, 13%) — large catalogs, lower 
  hit-rate, but high absolute hit counts.
- **Precision strategy:** Bad Bunny (22 songs → 22 hits, 100% hit rate, 
  85.4 avg popularity), Travis Scott (15 → 14, 93%) — small catalogs 
  where nearly every track is a hit.

Bad Bunny is the standout outlier: highest average popularity and a 
perfect hit-conversion rate in the top 15.

**Caveat — sampling bias:** the "22 songs" and "100%" figures reflect 
Bad Bunny's presence *within this dataset's sample*, not his full 
catalog (which contains hundreds of tracks). Since the dataset was built 
by sampling ~1,000 tracks per genre, it likely over-represents each 
artist's already-successful songs rather than capturing their complete 
discography. The 100% hit rate should be read as "the Bad Bunny tracks 
this dataset happened to sample are all hits" — not "everything Bad 
Bunny releases is a hit." This is the same type of dataset limitation 
already documented for `track_genre` (Q7).

**Business takeaway:** both volume and precision are viable paths to 
producing hits — worth distinguishing in the dashboard, but any 
artist-level popularity claim should be caveated by this sampling 
limitation.

## Q15 — Are explicit songs more or less popular?

In [22]:
query = """
SELECT explicit, 
       COUNT(*) AS songs, 
       SUM(CASE WHEN popularity >= 70 THEN 1 ELSE 0 END) AS hits, 
       ROUND(AVG(popularity), 1) AS popularity_avg
FROM (
  SELECT track_id, explicit, MAX(popularity) AS popularity
  FROM tracks
  GROUP BY track_id   
) AS unique_tracks
GROUP BY explicit
"""

pd.read_sql(query, conn)

,explicit,songs,hits,popularity_avg
0,0,81355,2567,32.9
1,1,7668,559,36.9




Deduplicated by track_id before aggregating.

**Finding:** Explicit tracks show both a higher average popularity 
(36.9 vs 32.9) and a notably higher hit-conversion rate (7.3% vs 3.2% 
— more than double).

**Caveat:** explicit content is unevenly distributed across genres 
(likely concentrated in rap, hip-hop, trap, reggaeton — genres already 
shown to overperform in Q11). This is likely a genre-confound rather 
than a direct causal effect of explicit content itself. Also note the 
sample imbalance: only 8.6% of unique tracks are explicit (7,668 of 
89,023), so the hit-rate percentage is more sensitive to individual hits.

